# Rendering Stromgren Ionization Front

This notebook render **particle-based snapshot** from SWIFT.

```text
SPH snapshot -> load_particles -> sph_to_grid -> grid_to_surface / grid_to_ridge_surface -> save(.hdf5) -> setup_animation in Blender
```

It is suited to surfaces extracted from SPH data, such as ionisation fronts, shells, or shock interfaces.


## 1. Building surfaces frame in Python 3.11 environment

This example uses the oxygen species helper documented in `Particle_Data.md`. `generate_species_fraction_fields()` returns a dict of getter functions that can be passed directly into `load_particles()`.

Two extraction paths are shown:

- `grid_to_surface()` for fields with a clean threshold.
- `grid_to_ridge_surface()` for filamentary or ridge-like structure.

The branching below mirrors the example described in the documentation: use the simpler isosurface path when possible, and the ridge extractor only where the field morphology requires it.


In [ ]:
import os
import yt

import sys
sys.path.append("../../..")

from AstroVis.backend import (
    generate_species_fraction_fields,
    load_particles,
    sph_to_grid,
    grid_to_surface,
    grid_to_ridge_surface,
    save,
)

input_dir = r"Data"
export_dir = r"Surface"
ptype = "PartType0"

os.makedirs(export_dir, exist_ok=True)

snapshots = sorted(
    f for f in os.listdir(input_dir)
    if f.endswith(".hdf5")
)

target_species = ["OI_fraction", "OII_fraction", "OIII_fraction", "OIV_fraction"]
surface_frames = {name: [] for name in target_species}

print(f"Found {len(snapshots)} snapshot(s).")

for frame_num, snapshot in enumerate(snapshots):
    ds = yt.load(os.path.join(input_dir, snapshot))
    oxygen_fields = generate_species_fraction_fields(ds, "O")

    particles = load_particles(
        ds,
        ptype=ptype,
        fields=oxygen_fields,
    )

    for species_name in target_species:
        grid = sph_to_grid(
            particles,
            fields=[species_name],
            res=256,
            intensive=True,
            center=True,
        )

        if species_name in {"OI_fraction", "OIV_fraction"}:
            surface = grid_to_surface(
                grid,
                threshold=0.5,
                field=species_name,
                plot_surface=(frame_num == 0),
                center=True,
                scale=1.0,
                path=os.path.join(export_dir, f"{species_name}_isosurface_frame{frame_num}.png")
            )
        else:
            surface = grid_to_ridge_surface(
                grid,
                field=species_name,
                sigma=1.2,
                lambda_pct=10,
                min_cluster_size=250,
                plot_check=(frame_num == 0),
                center=True,
                scale=1.0,
                path=os.path.join(export_dir, f"{species_name}_ridge_surface_frame{frame_num}.png")
            )

        if surface is not None:
            surface_frames[species_name].append(surface)

export_payload = {
    f"{species_name}_surface": frames
    for species_name, frames in surface_frames.items()
    if frames
}

save("Surface/oxygen_surfaces.hdf5", export_payload)


## 2. Blender setup

Run the cells below **inside Blender's Python environment**.

`setup_animation()` imports the HDF5 file and registers one animated mesh object per saved surface. Materials can then be created and applied by object name.


In [ ]:
from AstroVis.api import setup_animation, create_transparent_mesh_materials

data_path = r"C:\\Path\\To\\surface_exports"
objects = [
    "OI_fraction_surface",
    "OII_fraction_surface",
    "OIII_fraction_surface",
    "OIV_fraction_surface",
]

setup_animation(
    data_path,
    object=objects,
    center=False,
)

create_transparent_mesh_materials(objects, apply=True)


## Notes

- `save()` writes the surfaces into AstroVis' unified HDF5 scene format; this replaces the older `.npz` example flow.
- `center=True` during surface extraction recenters each extracted mesh around its own origin. If you want world-space motion preserved, keep the coordinate convention consistent across frames instead.
- For a simple opaque material, replace `create_transparent_mesh_materials()` with `create_mesh_materials()`.
